# CKD - LASSO + Rank Fusion

Real, raw UCI Chronic Kidney Disease data — same preprocessing as `ckd_anova.ipynb` (split-then-impute to avoid leakage, median imputation for lab values, mode imputation for categorical findings, mild-imbalance handling via `class_weight="balanced"`).

**What's different from `ckd_anova.ipynb`:** the mathematical operator feeding Rank Fusion is **LASSO** (L1-penalized Logistic Regression, absolute coefficient magnitude as the importance score) instead of ANOVA's F-statistic. `SklearnAdapter` already has an explicit branch for this (`elif hasattr(self.estimator, "coef_"): values = np.abs(self.estimator.coef_)` — labelled "Linear models / LASSO" in the package source), so no adapter changes were needed, only swapping which estimator gets wrapped.

In [1]:
# =============================================================================
# STEP 1: LOAD RAW DATASET
# =============================================================================

import pandas as pd
import numpy as np

df = pd.read_csv(
    "../../data/raw/ckd.csv"
)

df = df.drop(columns=["id"])

print("=" * 70)
print("RAW DATASET")
print("=" * 70)

print(f"Rows    : {df.shape[0]}")
print(f"Columns : {df.shape[1]}")

display(df.head())

RAW DATASET
Rows    : 400
Columns : 25


,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,...,pcv,wc,rc,htn,dm,cad,appet,pe,ane,classification
0,48.0,80.0,1.020,1.0,0.0,NaN,normal,notpresent,notpresent,121.0,...,44,7800,5.2,yes,yes,no,good,no,no,ckd
1,7.0,50.0,1.020,4.0,0.0,NaN,normal,notpresent,notpresent,NaN,...,38,6000,NaN,no,no,no,good,no,no,ckd
2,62.0,80.0,1.010,2.0,3.0,normal,normal,notpresent,notpresent,423.0,...,31,7500,NaN,no,yes,no,poor,no,yes,ckd
3,48.0,70.0,1.005,4.0,0.0,normal,abnormal,present,notpresent,117.0,...,32,6700,3.9,yes,no,no,poor,yes,yes,ckd
4,51.0,80.0,1.010,2.0,0.0,normal,normal,notpresent,notpresent,106.0,...,35,7300,4.6,no,no,no,good,no,no,ckd


In [2]:
# =============================================================================
# STEP 2: CLEAN KNOWN DATA-QUALITY ISSUES (before anything else)
# =============================================================================

print("=" * 70)
print("BEFORE CLEANUP")
print("=" * 70)
print("classification unique:", df["classification"].unique().tolist())
print("pcv dtype:", df["pcv"].dtype, "| wc dtype:", df["wc"].dtype, "| rc dtype:", df["rc"].dtype)

categorical_cols_raw = df.select_dtypes(include="object").columns
for col in categorical_cols_raw:
    df[col] = df[col].astype(str).str.strip()
    df[col] = df[col].replace({"nan": np.nan, "?": np.nan})

for col in ["pcv", "wc", "rc"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("\n" + "=" * 70)
print("AFTER CLEANUP")
print("=" * 70)
print("classification unique:", df["classification"].unique().tolist())
print("pcv dtype:", df["pcv"].dtype, "| wc dtype:", df["wc"].dtype, "| rc dtype:", df["rc"].dtype)

BEFORE CLEANUP
classification unique: ['ckd', 'ckd\t', 'notckd']
pcv dtype: str | wc dtype: str | rc dtype: str

AFTER CLEANUP
classification unique: ['ckd', 'notckd']
pcv dtype: float64 | wc dtype: float64 | rc dtype: float64


C:\Users\johnm\AppData\Local\Temp\ipykernel_4796\4219749122.py:11: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols_raw = df.select_dtypes(include="object").columns


In [3]:
# =============================================================================
# STEP 3: ENCODE CATEGORICAL FEATURES AND TARGET
# =============================================================================

target_column = "target"

binary_maps = {
    "rbc":   {"normal": 0, "abnormal": 1},
    "pc":    {"normal": 0, "abnormal": 1},
    "pcc":   {"notpresent": 0, "present": 1},
    "ba":    {"notpresent": 0, "present": 1},
    "htn":   {"no": 0, "yes": 1},
    "dm":    {"no": 0, "yes": 1},
    "cad":   {"no": 0, "yes": 1},
    "appet": {"poor": 0, "good": 1},
    "pe":    {"no": 0, "yes": 1},
    "ane":   {"no": 0, "yes": 1},
}

for col, mapping in binary_maps.items():
    df[col] = df[col].map(mapping)

df[target_column] = df["classification"].map({"ckd": 1, "notckd": 0})
df = df.drop(columns=["classification"])

print("=" * 70)
print("ENCODED DATASET")
print("=" * 70)
display(df.head())

print("\n" + "=" * 70)
print("TARGET DISTRIBUTION (0 = not CKD, 1 = CKD)")
print("=" * 70)
display(df[target_column].value_counts())
display((df[target_column].value_counts(normalize=True) * 100).round(2))

ENCODED DATASET


,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,...,pcv,wc,rc,htn,dm,cad,appet,pe,ane,target
0,48.0,80.0,1.020,1.0,0.0,NaN,0.0,0.0,0.0,121.0,...,44.0,7800.0,5.2,1.0,1.0,0.0,1.0,0.0,0.0,1
1,7.0,50.0,1.020,4.0,0.0,NaN,0.0,0.0,0.0,NaN,...,38.0,6000.0,NaN,0.0,0.0,0.0,1.0,0.0,0.0,1
2,62.0,80.0,1.010,2.0,3.0,0.0,0.0,0.0,0.0,423.0,...,31.0,7500.0,NaN,0.0,1.0,0.0,0.0,0.0,1.0,1
3,48.0,70.0,1.005,4.0,0.0,0.0,1.0,1.0,0.0,117.0,...,32.0,6700.0,3.9,1.0,0.0,0.0,0.0,1.0,1.0,1
4,51.0,80.0,1.010,2.0,0.0,0.0,0.0,0.0,0.0,106.0,...,35.0,7300.0,4.6,0.0,0.0,0.0,1.0,0.0,0.0,1



TARGET DISTRIBUTION (0 = not CKD, 1 = CKD)


target
1    250
0    150
Name: count, dtype: int64

target
1    62.5
0    37.5
Name: proportion, dtype: float64

In [4]:
# =============================================================================
# STEP 4: FEATURE AND TARGET SEPARATION
# =============================================================================

X = df.drop(target_column, axis=1)
y = df[target_column]

print("=" * 70)
print("FEATURE MATRIX (X)")
print("=" * 70)
print(f"Rows    : {X.shape[0]}")
print(f"Columns : {X.shape[1]}")

missing_summary = pd.DataFrame({
    "Missing Count": X.isnull().sum(),
    "Missing %": (X.isnull().sum() / len(X) * 100).round(2)
}).sort_values("Missing %", ascending=False)

print("\n" + "=" * 70)
print("MISSING VALUES BEFORE IMPUTATION")
print("=" * 70)
display(missing_summary[missing_summary["Missing Count"] > 0])

FEATURE MATRIX (X)
Rows    : 400
Columns : 24

MISSING VALUES BEFORE IMPUTATION


,Missing Count,Missing %
rbc,152,38.00
rc,131,32.75
wc,106,26.50
pot,88,22.00
sod,87,21.75
pcv,71,17.75
pc,65,16.25
hemo,52,13.00
su,49,12.25
sg,47,11.75


In [5]:
# =============================================================================
# STEP 5: TRAIN-TEST SPLIT (BEFORE imputation and scaling — avoids leakage)
# =============================================================================

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("=" * 70)
print("TRAIN-TEST SPLIT")
print("=" * 70)
print(f"X_train shape : {X_train.shape}")
print(f"X_test shape  : {X_test.shape}")

TRAIN-TEST SPLIT
X_train shape : (320, 24)
X_test shape  : (80, 24)


In [6]:
# =============================================================================
# STEP 6: IMPUTATION — fit on TRAINING data only, apply to both
# =============================================================================

from sklearn.impute import SimpleImputer

numerical_features = [
    "age", "bp", "sg", "al", "su", "bgr", "bu", "sc",
    "sod", "pot", "hemo", "pcv", "wc", "rc"
]

categorical_features = [
    "rbc", "pc", "pcc", "ba", "htn", "dm", "cad", "appet", "pe", "ane"
]

num_imputer = SimpleImputer(strategy="median")
cat_imputer = SimpleImputer(strategy="most_frequent")

num_imputer.fit(X_train[numerical_features])
cat_imputer.fit(X_train[categorical_features])

X_train_imputed = X_train.copy()
X_test_imputed = X_test.copy()

X_train_imputed[numerical_features] = num_imputer.transform(X_train[numerical_features])
X_test_imputed[numerical_features] = num_imputer.transform(X_test[numerical_features])

X_train_imputed[categorical_features] = cat_imputer.transform(X_train[categorical_features])
X_test_imputed[categorical_features] = cat_imputer.transform(X_test[categorical_features])

print("=" * 70)
print("IMPUTATION COMPLETE")
print("=" * 70)
print("Missing values remaining in X_train:", X_train_imputed.isnull().sum().sum())
print("Missing values remaining in X_test :", X_test_imputed.isnull().sum().sum())

IMPUTATION COMPLETE
Missing values remaining in X_train: 0
Missing values remaining in X_test : 0


In [7]:
# =============================================================================
# STEP 7: FEATURE SCALING (fit on TRAINING data only)
# =============================================================================

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train_imputed)
X_test_scaled = scaler.transform(X_test_imputed)

X_train_scaled = pd.DataFrame(X_train_scaled, columns=X_train_imputed.columns).reset_index(drop=True)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X_test_imputed.columns).reset_index(drop=True)

y_train = y_train.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

print("=" * 70)
print("SCALING COMPLETE")
print("=" * 70)
print(f"X_train_scaled shape : {X_train_scaled.shape}")
print(f"X_test_scaled shape  : {X_test_scaled.shape}")
display(X_train_scaled.head())

SCALING COMPLETE
X_train_scaled shape : (320, 24)
X_test_scaled shape  : (80, 24)


,age,bp,sg,al,su,rbc,pc,pcc,ba,bgr,...,hemo,pcv,wc,rc,htn,dm,cad,appet,pe,ane
0,-0.364027,0.224371,-0.508859,-0.678342,-0.347672,-0.356034,2.019744,-0.372545,-0.258199,-0.491252,...,-0.264295,-0.235403,0.708759,-0.650709,-0.75918,-0.684025,-0.291111,0.490214,-0.470504,-0.425220
1,0.463391,1.660345,-0.508859,2.411616,1.712604,-0.356034,-0.495112,-0.372545,-0.258199,1.577914,...,-1.894210,-2.324464,0.557632,-1.009720,1.31721,1.461935,3.435113,0.490214,-0.470504,2.351725
2,-0.364027,-1.211603,-1.444691,0.866637,-0.347672,-0.356034,2.019744,2.684237,-0.258199,1.759665,...,-0.916261,-1.218491,0.330943,0.067315,1.31721,1.461935,-0.291111,0.490214,-0.470504,-0.425220
3,1.054403,-1.211603,0.426974,-0.678342,-0.347672,-0.356034,-0.495112,-0.372545,-0.258199,0.403522,...,0.025467,0.133254,-0.084654,0.067315,1.31721,-0.684025,-0.291111,-2.039924,-0.470504,-0.425220
4,-0.186723,2.378332,-0.508859,1.639126,-0.347672,2.808717,-0.495112,2.684237,-0.258199,-0.505233,...,-1.423346,-1.587148,-1.255883,-2.685108,1.31721,-0.684025,3.435113,0.490214,-0.470504,2.351725


# LASSO

## 1. Baseline

In [8]:
# =============================================================================
# LASSO BASELINE FEATURE SELECTION (TOP-K = 5,10,15,20)
# =============================================================================

from sklearn.linear_model import LogisticRegression
import pandas as pd
import numpy as np

lasso_selector = LogisticRegression(
    penalty="l1",
    solver="liblinear",
    C=0.1,
    max_iter=2000,
    class_weight="balanced",
    random_state=42
)

lasso_selector.fit(X_train_scaled, y_train)

lasso_coefficients = lasso_selector.coef_[0]

lasso_scores = pd.DataFrame({
    "Feature": X_train_scaled.columns,
    "LASSO Coefficient": lasso_coefficients,
    "LASSO Score": np.abs(lasso_coefficients)
}).sort_values(by="LASSO Score", ascending=False).reset_index(drop=True)

k_values = [5, 10, 15, 20]
lasso_results = {}

for k in k_values:
    top_features = lasso_scores.head(k)["Feature"].tolist()
    mask = X_train_scaled.columns.isin(top_features)

    lasso_results[k] = {
        "features": top_features,
        "X_train": X_train_scaled.loc[:, mask],
        "X_test": X_test_scaled.loc[:, mask],
        "scores": lasso_scores.head(k)
    }

    print("=" * 70)
    print(f"TOP {k} LASSO FEATURES")
    print("=" * 70)
    print(top_features)
    print("\nScores:")
    print(lasso_scores.head(k))

TOP 5 LASSO FEATURES
['hemo', 'sg', 'htn', 'dm', 'pcv']

Scores:
  Feature  LASSO Coefficient  LASSO Score
0    hemo          -1.423788     1.423788
1      sg          -1.016467     1.016467
2     htn           0.427144     0.427144
3      dm           0.398986     0.398986
4     pcv          -0.255307     0.255307
TOP 10 LASSO FEATURES
['hemo', 'sg', 'htn', 'dm', 'pcv', 'al', 'rc', 'appet', 'pcc', 'pc']

Scores:
  Feature  LASSO Coefficient  LASSO Score
0    hemo          -1.423788     1.423788
1      sg          -1.016467     1.016467
2     htn           0.427144     0.427144
3      dm           0.398986     0.398986
4     pcv          -0.255307     0.255307
5      al           0.251402     0.251402
6      rc          -0.145765     0.145765
7   appet          -0.107240     0.107240
8     pcc           0.000000     0.000000
9      pc           0.000000     0.000000
TOP 15 LASSO FEATURES
['hemo', 'sg', 'htn', 'dm', 'pcv', 'al', 'rc', 'appet', 'pcc', 'pc', 'rbc', 'su', 'age', 'bp', 'ba'

c:\Users\johnm\msc_research\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\johnm\msc_research\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


In [9]:
# =============================================================================
# BASELINE MODELS
# =============================================================================

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=1000, class_weight="balanced", random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=100, class_weight="balanced", random_state=42
    ),
    "XGBoost": XGBClassifier(
        random_state=42,
        eval_metric="logloss",
        scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum()
    )
}

In [10]:
# =============================================================================
# MODEL EVALUATION
# =============================================================================

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)

results = []

for k in k_values:
    Xtr = lasso_results[k]["X_train"]
    Xts = lasso_results[k]["X_test"]
    features = ", ".join(lasso_results[k]["features"])

    for model_name, model in models.items():
        model.fit(Xtr, y_train)
        y_pred = model.predict(Xts)
        y_prob = model.predict_proba(Xts)[:, 1]

        results.append({
            "Method": "LASSO",
            "Top-K": k,
            "Model": model_name,
            "Selected Features": features,
            "Accuracy": accuracy_score(y_test, y_pred),
            "Precision": precision_score(y_test, y_pred, zero_division=0),
            "Recall": recall_score(y_test, y_pred, zero_division=0),
            "F1 Score": f1_score(y_test, y_pred, zero_division=0),
            "ROC-AUC": roc_auc_score(y_test, y_prob)
        })

results_df = pd.DataFrame(results)
display(results_df)

,Method,Top-K,Model,Selected Features,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,LASSO,5,Logistic Regression,"hemo, sg, htn, dm, pcv",0.9500,1.000000,0.92,0.958333,0.992667
1,LASSO,5,Random Forest,"hemo, sg, htn, dm, pcv",0.9750,1.000000,0.96,0.979592,0.999333
2,LASSO,5,XGBoost,"hemo, sg, htn, dm, pcv",0.9875,0.980392,1.00,0.990099,0.999333
3,LASSO,10,Logistic Regression,"hemo, sg, htn, dm, pcv, al, rc, appet, pcc, pc",0.9750,1.000000,0.96,0.979592,0.999667
4,LASSO,10,Random Forest,"hemo, sg, htn, dm, pcv, al, rc, appet, pcc, pc",0.9875,1.000000,0.98,0.989899,0.999667
5,LASSO,10,XGBoost,"hemo, sg, htn, dm, pcv, al, rc, appet, pcc, pc",0.9875,0.980392,1.00,0.990099,0.999667
6,LASSO,15,Logistic Regression,"hemo, sg, htn, dm, pcv, al, rc, appet, pcc, pc...",0.9750,1.000000,0.96,0.979592,0.998667
7,LASSO,15,Random Forest,"hemo, sg, htn, dm, pcv, al, rc, appet, pcc, pc...",0.9875,1.000000,0.98,0.989899,1.000000
8,LASSO,15,XGBoost,"hemo, sg, htn, dm, pcv, al, rc, appet, pcc, pc...",0.9875,0.980392,1.00,0.990099,1.000000
9,LASSO,20,Logistic Regression,"hemo, sg, htn, dm, pcv, al, rc, appet, pcc, pc...",0.9750,1.000000,0.96,0.979592,1.000000


In [11]:
results_df.to_csv(
    "../../../results/ckd/baseline_results/lasso_baseline_results.csv",
    index=False
)

## 2. Rank Fusion

In [12]:
import sys
import subprocess

subprocess.check_call([
    sys.executable,
    "-m", "pip", "install", "--upgrade", "--force-reinstall", "--no-cache-dir",
    "git+https://github.com/anandha-3679/DODA.git"
])

0

In [13]:
import sys
import doda
from doda import DODASelector
import inspect

print("Python:", sys.executable)
print("DODA:", doda.__file__)
print("Selector:", inspect.getfile(DODASelector))

Python: c:\Users\johnm\msc_research\.venv\Scripts\python.exe
DODA: c:\Users\johnm\msc_research\.venv\Lib\site-packages\doda\__init__.py
Selector: c:\Users\johnm\msc_research\.venv\Lib\site-packages\doda\selector.py


### Define RankFusion

Defined locally here so this notebook can actually run-

In [14]:
from doda.fusion.base import BaseFusion


class RankFusion(BaseFusion):
    """
    Rank-based fusion: combines the statistical (mathematical) ranking and
    the clinical-weight ranking by rank position rather than by raw score
    magnitude.
    """

    def fuse(self, math_scores, clinical_weights):

        math_ranked = sorted(
            math_scores.items(), key=lambda x: x[1], reverse=True
        )
        math_rank = {
            feature: idx + 1
            for idx, (feature, _) in enumerate(math_ranked)
        }

        clinical_ranked = sorted(
            clinical_weights.items(), key=lambda x: x[1], reverse=True
        )
        clinical_rank = {
            feature: idx + 1
            for idx, (feature, _) in enumerate(clinical_ranked)
        }

        n = len(math_scores)
        final_scores = {}

        for feature in math_scores:
            combined_rank_sum = (
                math_rank[feature]
                + clinical_rank.get(feature, n)
            )
            final_scores[feature] = (2 * n) - combined_rank_sum

        return final_scores

In [15]:
from doda import DODASelector

from doda.adapters import SklearnAdapter

from doda.knowledge.providers import JSONProvider

# NOTE: RankFusion imported from the LOCAL definition above, not from
# doda.fusion (it doesn't exist in the package yet — see markdown note).

In [16]:
# =============================================================================
# LASSO + RANK FUSION FEATURE SELECTION
# =============================================================================

import pandas as pd
from sklearn.linear_model import LogisticRegression

k_values = [5, 10, 15, 20]

provider = JSONProvider(
    "../../../config/clinical_weights/ckd_clinical_weights.json"
)

# Rank-based fusion instead of Hadamard multiplication
fusion = RankFusion()

rankfusion_results = {}

for k in k_values:

    print("\n" + "=" * 70)
    print(f"LASSO + RANK FUSION \u2014 TOP {k} FEATURES")
    print("=" * 70)

    lasso = SklearnAdapter(
        LogisticRegression(
            penalty="l1",
            solver="liblinear",
            C=0.1,
            max_iter=2000,
            class_weight="balanced",
            random_state=42
        )
    )

    selector = DODASelector(
        operators=[lasso],
        provider=provider,
        fusion=fusion,
        top_k=k
    )

    X_train_selected = selector.fit_transform(X_train_scaled, y_train)
    X_test_selected = selector.transform(X_test_scaled)

    selected_features = selector.get_selected_features()

    print("\nSelected Features:")
    print(selected_features)

    rankfusion_results[k] = {
        "selector": selector,
        "X_train": X_train_selected,
        "X_test": X_test_selected,
        "features": selected_features,
        "raw_math_scores": selector.raw_math_scores_,
        "math_scores": selector.math_scores_,
        "clinical_weights": selector.clinical_weights_,
        "final_scores": selector.final_scores_
    }

print("\n" + "=" * 70)
print("LASSO + RANK FUSION COMPLETED")
print("=" * 70)


LASSO + RANK FUSION — TOP 5 FEATURES
Provider inside knowledge engine: <doda.knowledge.providers.json_provider.JSONProvider object at 0x000002C230B20590>
DODA Pipeline Started
Running: LogisticRegression
DODA feature order:
['age', 'bp', 'sg', 'al', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr']
Number of features: 24
Number of coefficients: 24
Operator Scores:
{'LogisticRegression_1': {'age': np.float64(0.0), 'bp': np.float64(0.0), 'sg': np.float64(1.0164668375534045), 'al': np.float64(0.25140249503068596), 'su': np.float64(0.0), 'rbc': np.float64(0.0), 'pc': np.float64(0.0), 'pcc': np.float64(0.0), 'ba': np.float64(0.0), 'bgr': np.float64(0.0), 'bu': np.float64(0.0), 'sc': np.float64(0.0), 'sod': np.float64(0.0), 'pot': np.float64(0.0), 'hemo': np.float64(1.4237881705393525), 'pcv': np.float64(0.255307049278234), 'wc': np.float64(0.0), 'rc': np.float64(0.1457645315347347), 'htn': np.float64(0.4271435131441959), 'dm': np.float64(0.39898597769045857), 'cad': np.float64(0.0), 'appet': np.float

c:\Users\johnm\msc_research\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\johnm\msc_research\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
c:\Users\johnm\msc_research\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead o

## Cell 2 — Inspect Mathematical Scores

This remains exactly the same, because Rank Fusion does not change how the mathematical selector produces its scores.

In [17]:
k = 10

selector = rankfusion_results[k]["selector"]

raw_math_scores = pd.DataFrame(
    selector.raw_math_scores_.items(),
    columns=["Feature", "Raw Math Score"]
)

normalized_math_scores = pd.DataFrame(
    selector.math_scores_.items(),
    columns=["Feature", "Normalized Math Score"]
)

math_scores = raw_math_scores.merge(normalized_math_scores, on="Feature")
math_scores = math_scores.sort_values(by="Normalized Math Score", ascending=False)

print("=" * 70)
print("MATHEMATICAL SCORES")
print("=" * 70)
display(math_scores)

MATHEMATICAL SCORES


,Feature,Raw Math Score,Normalized Math Score
14,hemo,1.423788,1.000000
2,sg,1.016467,0.713917
18,htn,0.427144,0.300005
19,dm,0.398986,0.280228
15,pcv,0.255307,0.179315
3,al,0.251402,0.176573
17,rc,0.145765,0.102378
21,appet,0.107240,0.075320
7,pcc,0.000000,0.000000
6,pc,0.000000,0.000000


## Cell 3 — Inspect Clinical Weights

Again, this stays the same.

In [18]:
clinical_weights = pd.DataFrame(
    selector.clinical_weights_.items(),
    columns=["Feature", "Clinical Weight"]
)

clinical_weights = clinical_weights.sort_values(by="Clinical Weight", ascending=False)

print("=" * 70)
print("CLINICAL WEIGHTS")
print("=" * 70)
display(clinical_weights)

CLINICAL WEIGHTS


,Feature,Clinical Weight
3,al,1.0
11,sc,1.0
19,dm,0.8
5,rbc,0.8
18,htn,0.8
10,bu,0.7
1,bp,0.7
14,hemo,0.7
13,pot,0.7
23,ane,0.6


## Cell 4 — Inspect Final Rank Fusion Scores

Labelled **Final Rank Fusion Score** rather than a generic final score, since this is specifically testing the fusion mechanism.

In [19]:
final_scores = pd.DataFrame(
    selector.final_scores_.items(),
    columns=["Feature", "Final Rank Fusion Score"]
)

final_scores = final_scores.sort_values(by="Final Rank Fusion Score", ascending=False)

print("=" * 70)
print("FINAL RANK FUSION SCORES")
print("=" * 70)
display(final_scores)

FINAL RANK FUSION SCORES


,Feature,Final Rank Fusion Score
3,al,41
18,htn,41
19,dm,39
14,hemo,38
5,rbc,33
1,bp,32
15,pcv,30
0,age,29
2,sg,29
11,sc,28


In [20]:
selector.get_selected_features()

['al', 'htn', 'dm', 'hemo', 'rbc', 'bp', 'pcv', 'age', 'sg', 'sc']

In [21]:
# =============================================================================
# COMPARE RANKINGS
# =============================================================================

comparison = selector.compare_scores()

display(comparison)


MATHEMATICAL vs CLINICAL vs RANKFUSION

Fusion Method: RankFusion

Top Features After Fusion:
  Feature  Math Rank  Clinical Rank  Final Rank  Rank Change
0      al          6              1           1            5
1     htn          3              4           2            1
2      dm          4              5           3            1
3    hemo          1              9           4           -3
4     rbc         12              3           5            7
5      bp         10              6           6            4
6     pcv          5             13           7           -2
7     age          9             10           8            1
8      sg          2             17           9           -7
9      sc         18              2          10            8


,Feature,Math Rank,Normalized Math Score,Clinical Rank,Clinical Weight,Final Rank,Final Score,Rank Change
0,al,6,0.1766,1,1.0,1,41,5
1,htn,3,0.3000,4,0.8,2,41,1
2,dm,4,0.2802,5,0.8,3,39,1
3,hemo,1,1.0000,9,0.7,4,38,-3
4,rbc,12,0.0000,3,0.8,5,33,7
5,bp,10,0.0000,6,0.7,6,32,4
6,pcv,5,0.1793,13,0.6,7,30,-2
7,age,9,0.0000,10,0.6,8,29,1
8,sg,2,0.7139,17,0.5,9,29,-7
9,sc,18,0.0000,2,1.0,10,28,8


## Cell 7 — Model Evaluation

Same evaluation structure, but use `rankfusion_results`.

In [22]:
# =============================================================================
# RANK FUSION MODEL EVALUATION
# =============================================================================

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
)

rankfusion_model_results = []

for k in k_values:

    Xtr = rankfusion_results[k]["X_train"]
    Xts = rankfusion_results[k]["X_test"]
    features = ", ".join(rankfusion_results[k]["features"])

    for model_name, model in models.items():

        model.fit(Xtr, y_train)
        y_pred = model.predict(Xts)
        y_prob = model.predict_proba(Xts)[:, 1]

        rankfusion_model_results.append({
            "Method": "LASSO + Rank Fusion",
            "Top-K": k,
            "Model": model_name,
            "Selected Features": features,
            "Accuracy": accuracy_score(y_test, y_pred),
            "Precision": precision_score(y_test, y_pred, zero_division=0),
            "Recall": recall_score(y_test, y_pred, zero_division=0),
            "F1 Score": f1_score(y_test, y_pred, zero_division=0),
            "ROC-AUC": roc_auc_score(y_test, y_prob)
        })

rankfusion_results_df = pd.DataFrame(rankfusion_model_results)
display(rankfusion_results_df)

,Method,Top-K,Model,Selected Features,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,LASSO + Rank Fusion,5,Logistic Regression,"al, htn, dm, hemo, rbc",0.9375,1.000000,0.90,0.947368,0.987667
1,LASSO + Rank Fusion,5,Random Forest,"al, htn, dm, hemo, rbc",0.9625,0.979592,0.96,0.969697,0.988333
2,LASSO + Rank Fusion,5,XGBoost,"al, htn, dm, hemo, rbc",0.9625,0.979592,0.96,0.969697,0.996667
3,LASSO + Rank Fusion,10,Logistic Regression,"al, htn, dm, hemo, rbc, bp, pcv, age, sg, sc",0.9750,1.000000,0.96,0.979592,0.998667
4,LASSO + Rank Fusion,10,Random Forest,"al, htn, dm, hemo, rbc, bp, pcv, age, sg, sc",1.0000,1.000000,1.00,1.000000,1.000000
5,LASSO + Rank Fusion,10,XGBoost,"al, htn, dm, hemo, rbc, bp, pcv, age, sg, sc",1.0000,1.000000,1.00,1.000000,1.000000
6,LASSO + Rank Fusion,15,Logistic Regression,"al, htn, dm, hemo, rbc, bp, pcv, age, sg, sc, ...",0.9750,1.000000,0.96,0.979592,0.998667
7,LASSO + Rank Fusion,15,Random Forest,"al, htn, dm, hemo, rbc, bp, pcv, age, sg, sc, ...",1.0000,1.000000,1.00,1.000000,1.000000
8,LASSO + Rank Fusion,15,XGBoost,"al, htn, dm, hemo, rbc, bp, pcv, age, sg, sc, ...",1.0000,1.000000,1.00,1.000000,1.000000
9,LASSO + Rank Fusion,20,Logistic Regression,"al, htn, dm, hemo, rbc, bp, pcv, age, sg, sc, ...",0.9750,1.000000,0.96,0.979592,1.000000


In [23]:
import os

os.makedirs("../../../results/ckd/rankfusion_results", exist_ok=True)
rankfusion_results_df.to_csv(
    "../../../results/ckd/rankfusion_results/lasso_rankfusion_results.csv",
    index=False
)

In [24]:
# =============================================================================
# COMBINED COMPARISON
# =============================================================================

comparison_df = pd.concat(
    [results_df, rankfusion_results_df],
    ignore_index=True
)

display(comparison_df)

,Method,Top-K,Model,Selected Features,Accuracy,Precision,Recall,F1 Score,ROC-AUC
0,LASSO,5,Logistic Regression,"hemo, sg, htn, dm, pcv",0.9500,1.000000,0.92,0.958333,0.992667
1,LASSO,5,Random Forest,"hemo, sg, htn, dm, pcv",0.9750,1.000000,0.96,0.979592,0.999333
2,LASSO,5,XGBoost,"hemo, sg, htn, dm, pcv",0.9875,0.980392,1.00,0.990099,0.999333
3,LASSO,10,Logistic Regression,"hemo, sg, htn, dm, pcv, al, rc, appet, pcc, pc",0.9750,1.000000,0.96,0.979592,0.999667
4,LASSO,10,Random Forest,"hemo, sg, htn, dm, pcv, al, rc, appet, pcc, pc",0.9875,1.000000,0.98,0.989899,0.999667
5,LASSO,10,XGBoost,"hemo, sg, htn, dm, pcv, al, rc, appet, pcc, pc",0.9875,0.980392,1.00,0.990099,0.999667
6,LASSO,15,Logistic Regression,"hemo, sg, htn, dm, pcv, al, rc, appet, pcc, pc...",0.9750,1.000000,0.96,0.979592,0.998667
7,LASSO,15,Random Forest,"hemo, sg, htn, dm, pcv, al, rc, appet, pcc, pc...",0.9875,1.000000,0.98,0.989899,1.000000
8,LASSO,15,XGBoost,"hemo, sg, htn, dm, pcv, al, rc, appet, pcc, pc...",0.9875,0.980392,1.00,0.990099,1.000000
9,LASSO,20,Logistic Regression,"hemo, sg, htn, dm, pcv, al, rc, appet, pcc, pc...",0.9750,1.000000,0.96,0.979592,1.000000


In [25]:
for k in k_values:

    selector = rankfusion_results[k]["selector"]

    raw_rank = [
        x[0] for x in sorted(
            selector.raw_math_scores_.items(), key=lambda x: x[1], reverse=True
        )
    ]

    rankfusion_rank = [
        x[0] for x in sorted(
            selector.final_scores_.items(), key=lambda x: x[1], reverse=True
        )
    ]

    print("\n" + "=" * 70)
    print(f"TOP {k}")
    print("=" * 70)

    print("Raw Math Ranking:")
    print(raw_rank)

    print("\nRank Fusion Ranking:")
    print(rankfusion_rank)

    print("\nTop-K Same:", raw_rank[:k] == rankfusion_rank[:k])


TOP 5
Raw Math Ranking:
['hemo', 'sg', 'htn', 'dm', 'pcv', 'al', 'rc', 'appet', 'age', 'bp', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr', 'bu', 'sc', 'sod', 'pot', 'wc', 'cad', 'pe', 'ane']

Rank Fusion Ranking:
['al', 'htn', 'dm', 'hemo', 'rbc', 'bp', 'pcv', 'age', 'sg', 'sc', 'bu', 'rc', 'bgr', 'pot', 'su', 'appet', 'sod', 'pc', 'pcc', 'cad', 'ba', 'pe', 'ane', 'wc']

Top-K Same: False

TOP 10
Raw Math Ranking:
['hemo', 'sg', 'htn', 'dm', 'pcv', 'al', 'rc', 'appet', 'age', 'bp', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr', 'bu', 'sc', 'sod', 'pot', 'wc', 'cad', 'pe', 'ane']

Rank Fusion Ranking:
['al', 'htn', 'dm', 'hemo', 'rbc', 'bp', 'pcv', 'age', 'sg', 'sc', 'bu', 'rc', 'bgr', 'pot', 'su', 'appet', 'sod', 'pc', 'pcc', 'cad', 'ba', 'pe', 'ane', 'wc']

Top-K Same: False

TOP 15
Raw Math Ranking:
['hemo', 'sg', 'htn', 'dm', 'pcv', 'al', 'rc', 'appet', 'age', 'bp', 'su', 'rbc', 'pc', 'pcc', 'ba', 'bgr', 'bu', 'sc', 'sod', 'pot', 'wc', 'cad', 'pe', 'ane']

Rank Fusion Ranking:
['al', 'htn', 'dm',